# 00 — Download, select, and build district graphs

Prepare the daily OD inputs and district networks used by the paper. This notebook follows four stages: configuration, daily acquisition and selection, inspection of the selected sample, and network construction. Each stage shows its relevant outputs and checks.

The sample retains positive expanded trip weights with nonmissing income, age, and gender and one of the three valid income bands. Residence is preserved, including missing values, because notebook 02 conditions its geographic benchmark on recorded residence. Trip weights represent expanded counts supplied by the provider.

Run **00 → 01 → 02**, each from a fresh kernel. Notebook 01 prepares the geography; notebook 02 computes the paper results. Daily inputs and graph files are saved locally for the next stages.

## 1. Configuration

Load the package, define the four study weeks, and choose the input/output folders.

In [1]:
import hashlib
import json
import os
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyspainmobility as psm
from IPython.display import display
from scipy.sparse import save_npz

if not hasattr(psm, "select_od"):
    raise ImportError("Use the current pySpainMobility selection/network API.")

pd.set_option("display.max_colwidth", None)
if psm.__version__ != "2.1.2":
    raise ImportError("This paper workflow requires pySpainMobility 2.1.2; install requirements.txt.")


In [2]:
notebook_names = ["00_download_and_build_graphs.ipynb", "01_prepare_analysis_inputs.ipynb", "02_main_paper_analysis.ipynb"]
configured_paper = os.environ.get("PYSPAIN_PAPER_DIR")
if configured_paper:
    PAPER_DIR = Path(configured_paper).expanduser().resolve()
else:
    PAPER_DIR = next(
        (candidate for parent in [Path.cwd(), *Path.cwd().parents]
         for candidate in [parent, parent / "examples/paper"]
         if all((candidate / name).exists() for name in notebook_names)), None,
    )
    if PAPER_DIR is None:
        raise FileNotFoundError("Open these notebooks from examples/paper or set PYSPAIN_PAPER_DIR.")

PIPELINE_OUTPUT = Path(os.environ.get("PYSPAIN_PIPELINE_OUTPUT", str(PAPER_DIR / "outputs/paper_inputs"))).expanduser().resolve()
RAW_DATA_DIR = Path(os.environ.get("PYSPAIN_RAW_DATA_DIR", str(PAPER_DIR / "raw_data"))).expanduser().resolve()
WINDOWS = {
    "winter": ("2023-01-16", "2023-01-22"),
    "spring": ("2023-05-08", "2023-05-14"),
    "summer": ("2023-07-17", "2023-07-23"),
    "autumn": ("2023-10-09", "2023-10-15"),
}
MAIN = list(WINDOWS)
BANDS = {"<10": "low", "10-15": "mid", ">15": "high"}
GROUPS = list(BANDS.values())
PROVINCES = [f"{number:02d}" for number in range(1, 53)]
print("Notebook directory:", PAPER_DIR)
print("Prepared inputs:", PIPELINE_OUTPUT)

CACHE_DIR = PIPELINE_OUTPUT / "cache"
GRAPH_DIR = PIPELINE_OUTPUT / "graphs"
for directory in [CACHE_DIR, GRAPH_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
FORCE_REPARSE = os.environ.get("PYSPAIN_FORCE_REPARSE", "0") == "1"
DIMENSIONS = ["income", "age", "gender", "residence_province_ine_code"]
KEYS = ["date", "id_origin", "id_destination", "income", "residence_province"]
package_directory = Path(psm.__file__).resolve().parent
package_hash = hashlib.sha256(b"".join(path.read_bytes() for path in sorted(package_directory.rglob("*.py")))).hexdigest()
reuse_path = PIPELINE_OUTPUT
REUSE_DIR = Path(os.environ.get("PYSPAIN_CACHE_REUSE_DIR", str(reuse_path))).expanduser().resolve()
display(pd.DataFrame.from_dict(WINDOWS, orient="index", columns=["Start", "End"]))
print("Loaded package:", psm.__version__, psm.__file__)


Notebook directory: .
Prepared inputs: outputs/paper_inputs


,Start,End
winter,2023-01-16,2023-01-22
spring,2023-05-08,2023-05-14
summer,2023-07-17,2023-07-23
autumn,2023-10-09,2023-10-15


Loaded package: 2.1.2 ../../pyspainmobility/__init__.py


In [3]:
def digest(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()


def show_summary(title, table, decimals=2):
    print(title)
    display(table.round(decimals))


## 2. Daily acquisition and demographic selection

Each daily cache contains departure district, destination district, date, income band, residence province, expanded trip count, and trip-km. Age and gender are used for complete-case selection before grouping. Residence codes are standardized to 01–52; unknown residence remains missing.

Validated caches are reused by default after package/source fingerprint and checksum checks. Otherwise, `Mobility.get_od_data()` acquires the daily data and the public selection API groups the selected rows. Set `PYSPAIN_FORCE_REPARSE=1` to rebuild through Mobility; uncached acquisition requires access to the Ministry server.

### Cache validation

Reuse is recorded separately from acquisition through the public API.

In [4]:
def matching_cache(day, source_hash):
    if FORCE_REPARSE:
        return None
    for directory in [CACHE_DIR, REUSE_DIR / "cache"]:
        for sidecar in sorted(directory.glob(f"{day}_*.json")):
            record = json.loads(sidecar.read_text())
            path = sidecar.with_suffix(".parquet")
            if (record.get("sample_schema") != "complete_demographic_residence_v1"
                    or record.get("package_sha256") != package_hash
                    or (source_hash and record.get("source_sha256") != source_hash)
                    or not path.exists()):
                continue
            if digest(path) != record.get("cache_sha256"):
                raise ValueError(f"Cache integrity failure: {path}")
            target = CACHE_DIR / path.name
            if path.resolve() != target.resolve():
                shutil.copy2(path, target)
            record.update(cache=str(target), acquisition_mode="verified_cache")
            target.with_suffix(".json").write_text(json.dumps(record, indent=2))
            return target, record
    return None


### Acquisition and selection

Apply the same selection to every day and preserve the recorded residence information.

In [5]:
def acquire_day(day):
    filename = f"{day.replace('-', '')}_Viajes_distritos_v2.csv.gz"
    source = RAW_DATA_DIR / filename
    source_hash = digest(source) if source.exists() else None
    cached = matching_cache(day, source_hash)
    if cached is not None:
        return cached

    work = PIPELINE_OUTPUT / "acquisition" / day
    work.mkdir(parents=True, exist_ok=True)
    local = work / filename
    if source.exists():
        if local.exists() and digest(local) != source_hash:
            local.unlink()
        if not local.exists():
            try:
                local.hardlink_to(source)
            except OSError:
                shutil.copy2(source, local)
    mobility = psm.Mobility(version=2, zones="districts", start_date=day, end_date=day,
                            output_directory=str(work), backend="polars")
    mobility.get_od_data(dimensions=DIMENSIONS, return_df=False, allow_partial=False)
    processed = list(work.glob("*.parquet"))
    if len(processed) != 1:
        raise ValueError(f"Expected one processed daily table: {processed}")
    frame = pl.read_parquet(processed[0])
    complete = frame.filter(pl.all_horizontal(pl.col(column).is_not_null() for column in ["income", "age", "gender"]))
    selected = complete.filter(pl.col("income").is_in(list(BANDS)), pl.col("n_trips") > 0)
    selected = selected.with_columns(
        pl.when(pl.col("residence_province_ine_code").cast(pl.String).str.contains(r"^\d{1,2}$"))
        .then(pl.col("residence_province_ine_code").cast(pl.String).str.zfill(2))
        .otherwise(None).alias("residence_province"))
    selected = selected.with_columns(pl.when(pl.col("residence_province").is_in(PROVINCES))
                                    .then(pl.col("residence_province")).otherwise(None).alias("residence_province"))
    reduced = psm.select_od(selected, group_by=KEYS).collect()
    assert reduced.height > 0 and reduced["date"].cast(pl.String).unique().to_list() == [day]
    source_hash = digest(local)
    target = CACHE_DIR / f"{day}_{source_hash[:12]}_{package_hash[:12]}.parquet"
    reduced.write_parquet(target)
    manifest = mobility.get_acquisition_manifest()
    manifest.to_csv(work / "manifest.csv", index=False)
    record = {
        "date": day, "sample_schema": "complete_demographic_residence_v1",
        "source": str(source if source.exists() else local), "source_sha256": source_hash,
        "package_sha256": package_hash, "cache": str(target), "cache_sha256": digest(target),
        "input_rows": len(frame), "input_weight": float(frame["n_trips"].sum()),
        "complete_demographic_rows": len(complete), "complete_demographic_weight": float(complete["n_trips"].sum()),
        "retained_weight": float(reduced["n_trips"].sum()), "acquisition_mode": "public_api",
        "manifest": json.loads(manifest.to_json(orient="records", date_format="iso")),
    }
    target.with_suffix(".json").write_text(json.dumps(record, indent=2))
    for path in processed:
        path.unlink()
    return target, record


### Prepare and check all 28 days

In [6]:
provenance = {
    "package_version": psm.__version__, "package_sha256": package_hash,
    "notebook00_sha256": digest(PAPER_DIR / "00_download_and_build_graphs.ipynb"),
    "sample": "income, age and gender all nonmissing; income in three valid bands; positive trips",
    "dimensions": DIMENSIONS, "windows": WINDOWS, "main_windows": MAIN,
    "distance_method": "EPSG:25830 representative points, Euclidean km; source trip-km supplementary",
    "sources": [], "inputs_status": "acquiring",
}
for season, window in WINDOWS.items():
    for day in pd.date_range(*window).strftime("%Y-%m-%d"):
        path, record = acquire_day(day)
        schema = pl.read_parquet_schema(path)
        assert set(KEYS + ["n_trips", "trips_total_length_km"]).issubset(schema)
        data = pl.scan_parquet(path)
        check = data.select(pl.col("n_trips").sum(), pl.col("income").is_in(list(BANDS)).all().alias("valid_income"),
                            (pl.col("n_trips") > 0).all().alias("positive_weight")).collect().row(0)
        np.testing.assert_allclose(check[0], record["retained_weight"], rtol=1e-9)
        assert check[1] and check[2]
        provenance["sources"].append({"window": season, **record})
        (PIPELINE_OUTPUT / "provenance.json").write_text(json.dumps(provenance, indent=2))
        print(season, day, record["acquisition_mode"], flush=True)
assert len(provenance["sources"]) == 28
provenance["inputs_status"] = "acquired"
(PIPELINE_OUTPUT / "provenance.json").write_text(json.dumps(provenance, indent=2))
daily_inventory = pd.DataFrame(provenance["sources"])[["window", "date", "retained_weight", "acquisition_mode"]]
daily_inventory.to_csv(PIPELINE_OUTPUT / "daily_input_inventory.csv", index=False)

print(f"Prepared {len(provenance['sources'])} daily inputs across {len(WINDOWS)} study weeks.")
show_summary("Daily selected weights and acquisition mode", daily_inventory.assign(
    retained_weight=daily_inventory.retained_weight / 1e6
).rename(columns={"window": "Season", "date": "Date",
                  "retained_weight": "Selected trips (million)", "acquisition_mode": "Acquisition mode"}))
print("Daily schemas, valid income bands, positive weights, and retained-volume checks: passed.")


winter 2023-01-16 verified_cache


winter 2023-01-17 verified_cache


winter 2023-01-18 verified_cache


winter 2023-01-19 verified_cache


winter 2023-01-20 verified_cache


...

autumn 2023-10-14 verified_cache


autumn 2023-10-15 verified_cache


Prepared 28 daily inputs across 4 study weeks.
Daily selected weights and acquisition mode


,Season,Date,Selected trips (million),Acquisition mode
0,winter,2023-01-16,83.70,verified_cache
1,winter,2023-01-17,84.19,verified_cache
2,winter,2023-01-18,85.25,verified_cache
3,winter,2023-01-19,86.32,verified_cache
4,winter,2023-01-20,88.24,verified_cache
5,winter,2023-01-21,73.72,verified_cache
6,winter,2023-01-22,63.26,verified_cache
7,spring,2023-05-08,88.82,verified_cache
8,spring,2023-05-09,89.48,verified_cache
9,spring,2023-05-10,90.31,verified_cache


Daily schemas, valid income bands, positive weights, and retained-volume checks: passed.


## 3. Inspect the selected sample

Inspect grouped OD rows before geographic filtering or removal of same-district trips. The following summaries compare released and selected trip weights, then show the three income layers and the proportion of their weight with recorded residence. Missing residence remains in this sample.

In [7]:
example = pl.scan_parquet(provenance["sources"][0]["cache"]).select(
    *KEYS, "n_trips", "trips_total_length_km"
).head(6).collect().to_pandas()
show_summary("Example grouped OD rows from the first study day", example.rename(columns={
    "date": "Date", "id_origin": "Departure district", "id_destination": "Destination district",
    "income": "Income band", "residence_province": "Residence province",
    "n_trips": "Expanded trips", "trips_total_length_km": "Expanded trip-km",
}))

selection_rows = []
for season in MAIN:
    records = [row for row in provenance["sources"] if row["window"] == season]
    released = sum(row["input_weight"] for row in records)
    complete = sum(row["complete_demographic_weight"] for row in records)
    retained = sum(row["retained_weight"] for row in records)
    selection_rows.append({
        "Season": season.title(), "Days": len(records), "Released trips (million)": released / 1e6,
        "Complete-demographic trips (million)": complete / 1e6,
        "Selected trips (million)": retained / 1e6, "Selected weight (%)": 100 * retained / released,
    })
show_summary("Selection audit before geographic filtering and loop removal", pd.DataFrame(selection_rows))


Example grouped OD rows from the first study day


,Date,Departure district,Destination district,Income band,Residence province,Expanded trips,Expanded trip-km
0,2023-01-16,4808404,4804401,<10,48,1.70,8.50
1,2023-01-16,36023,PT111,10-15,03,1.00,2.45
2,2023-01-16,3120101,3120102,>15,20,7.75,5.54
3,2023-01-16,04064,0404902,>15,48,3.30,8.59
4,2023-01-16,2807909,2807919,10-15,03,2.33,27.99
5,2023-01-16,3803803,38032,<10,38,71.92,571.33


Selection audit before geographic filtering and loop removal


,Season,Days,Released trips (million),Complete-demographic trips (million),Selected trips (million),Selected weight (%)
0,Winter,7,891.67,564.68,564.68,63.33
1,Spring,7,947.49,589.00,589.00,62.16
2,Summer,7,893.66,546.42,546.42,61.14
3,Autumn,7,898.46,558.13,558.13,62.12


In [8]:
income_rows = []
for season in MAIN:
    paths = [row["cache"] for row in provenance["sources"] if row["window"] == season]
    layers = pl.scan_parquet(paths).group_by("income").agg(
        pl.col("n_trips").sum().alias("weight"),
        pl.col("n_trips").filter(pl.col("residence_province").is_not_null()).sum().alias("known_residence_weight"),
    ).collect().to_pandas().set_index("income")
    for band, group in BANDS.items():
        row = layers.loc[band]
        income_rows.append({
            "Season": season.title(), "Income": {"low": "Low", "mid": "Middle", "high": "High"}[group],
            "Selected trips (million)": row.weight / 1e6,
            "Share of selected weight (%)": 100 * row.weight / layers.weight.sum(),
            "Known residence weight (%)": 100 * row.known_residence_weight / row.weight,
        })
show_summary("Income-layer volumes and residence coverage", pd.DataFrame(income_rows))


Income-layer volumes and residence coverage


,Season,Income,Selected trips (million),Share of selected weight (%),Known residence weight (%)
0,Winter,Low,84.51,14.97,100.0
1,Winter,Middle,354.86,62.84,100.0
2,Winter,High,125.32,22.19,100.0
3,Spring,Low,133.65,22.69,100.0
4,Spring,Middle,339.07,57.57,100.0
5,Spring,High,116.28,19.74,100.0
6,Summer,Low,82.18,15.04,100.0
7,Summer,Middle,343.68,62.90,100.0
8,Summer,High,120.55,22.06,100.0
9,Autumn,Low,83.74,15.00,100.0


## 4. Sparse district networks and graph audit

Networks use the same fixed district node index across weeks and income layers. Same-district trips are removed. Geographic matching and removal of within-province trips occur later, so these graph weights must not be confused with retained inter-province weights.

Save CSR matrices with their ordered node identifiers. The daily OD caches remain the input for notebook 02 because aggregate graphs cannot recover residence-conditioned information.

In [9]:
all_paths = [record["cache"] for record in provenance["sources"]]
ids = pl.scan_parquet(all_paths).select("id_origin", "id_destination").collect()
node_ids = sorted(set(ids["id_origin"].to_list()) | set(ids["id_destination"].to_list()))
node_index = psm.NodeIndex(node_ids, zoning_id="MITMA_districts_source", zoning_version="2")
del ids
(GRAPH_DIR / "node_index.json").write_text(json.dumps({"node_ids": node_ids, "zoning_id": "MITMA_districts_source", "zoning_version": "2"}, indent=2))

print(f"Shared source district index: {len(node_ids):,} identifiers.")
print("The fixed index includes every district appearing as an endpoint across the four weeks.")


Shared source district index: 3,857 identifiers.
The fixed index includes every district appearing as an endpoint across the four weeks.


In [10]:
inventory_rows = []
network_audits = {}
for season in MAIN:
    paths = [record["cache"] for record in provenance["sources"] if record["window"] == season]
    season_dir = GRAPH_DIR / season
    season_dir.mkdir(exist_ok=True)
    for group in ["all", *GROUPS]:
        selected = pl.scan_parquet(paths)
        if group != "all":
            income_code = next(code for code, label in BANDS.items() if label == group)
            selected = psm.select_od(selected, filters={"income": income_code})
        network = psm.build_network(selected, spec=psm.NetworkSpec(self_loops="drop"), node_index=node_index)
        save_npz(season_dir / f"{group}.npz", network.adjacency)
        strength = psm.node_strengths(network).to_pandas()
        active_nodes = int(((strength.in_strength + strength.out_strength) > 0).sum())
        inventory_rows.append({"season": season, "group": group, "fixed_nodes": len(node_ids),
                               "active_nodes": active_nodes, "directed_edges": network.adjacency.nnz,
                               "graph_weight": network.total_weight})
        network_audits[f"{season}/{group}"] = network.audit()
        np.testing.assert_allclose(strength.in_strength.sum(), network.total_weight, rtol=1e-9)
        np.testing.assert_allclose(strength.out_strength.sum(), network.total_weight, rtol=1e-9)
        del network, strength
inventory = pd.DataFrame(inventory_rows)
inventory.to_csv(PIPELINE_OUTPUT / "graph_inventory.csv", index=False)
(PIPELINE_OUTPUT / "district_network_audits.json").write_text(json.dumps(network_audits, indent=2, default=str))

print("Incoming and outgoing strengths reproduce each network's total weight: passed.")


Incoming and outgoing strengths reproduce each network's total weight: passed.


In [11]:
show_summary("District graph inventory after same-district trip removal", inventory.assign(
    graph_weight=inventory.graph_weight / 1e6
).rename(columns={"season": "Season", "group": "Layer", "fixed_nodes": "Fixed nodes",
                  "active_nodes": "Active nodes", "directed_edges": "Directed edges",
                  "graph_weight": "Expanded trip weight (million)"}))


District graph inventory after same-district trip removal


,Season,Layer,Fixed nodes,Active nodes,Directed edges,Expanded trip weight (million)
0,winter,all,3857,3854,837839,386.00
1,winter,low,3857,3843,234124,51.55
2,winter,mid,3857,3854,671285,249.31
3,winter,high,3857,3854,311695,85.15
4,spring,all,3857,3854,976563,401.45
5,spring,low,3857,3854,327850,82.31
6,spring,mid,3857,3854,749131,240.72
7,spring,high,3857,3854,355749,78.42
8,summer,all,3857,3856,1123523,377.24
9,summer,low,3857,3854,281742,49.94


## 5. Outputs for notebook 01

Daily caches preserve the OD and residence information needed by the analysis. The sparse graph files and shared node index document the district networks; their weights still include trips within provinces. Notebook 01 prepares the validated geographic correspondence and the inter-province filtering audit.

In [12]:
show_summary("Files prepared by notebook 00", pd.DataFrame([
    {"Output": "Daily OD caches", "Files": len(provenance["sources"]), "Location": str(CACHE_DIR)},
    {"Output": "District network matrices", "Files": len(inventory), "Location": str(GRAPH_DIR)},
    {"Output": "Acquisition provenance", "Files": 1, "Location": str(PIPELINE_OUTPUT / "provenance.json")},
]))
print("Inputs acquired. Next: run 01_prepare_analysis_inputs.ipynb.")


Files prepared by notebook 00


,Output,Files,Location
0,Daily OD caches,28,outputs/paper_inputs/cache
1,District network matrices,16,outputs/paper_inputs/graphs
2,Acquisition provenance,1,outputs/paper_inputs/provenance.json


Inputs acquired. Next: run 01_prepare_analysis_inputs.ipynb.
